## Import my modules

In [54]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

## create connection to jumia website

In [51]:
URL = "https://www.jumia.co.ke"

response = requests.get(URL)

print(response.status_code)

200


## use `BeautifulSoup` to pull information from website

In [ ]:
# create an object called soup that stores everything in that website page
soup = BeautifulSoup(response.content, 'html.parser')

# i want only the section that has Flash Sales
flash_header = soup.find('h2', string=re.compile('Flash Sales', re.I))

#find the parent container for the flash sales section header
section = flash_header.find_parent('section')

#extract all products with the section
products = section.find_all('article', class_='prd')

## Extract the needed product informations and store in a dataframe

In [57]:
# create an empty list to store the data
datasets = []

# iterate through all product list to pick information
for indexes,product in enumerate(products):
    # print(indexes,product)

    card = products[indexes]
    name_elem = card.find('div', class_='name')
    name = name_elem.get_text(strip=True) if name_elem else "N/A"
    # print(name)

    link = card.find('a', class_='core')
    brand = link.get('data-ga4-item_brand', 'N/A') if link else "N/A"
    # print(brand)

    price_elem = card.find('div', class_='prc')
    price_text = price_elem.get_text(strip=True)
    price_match = re.search(r'KSh\s*([\d,]+)', price_text)
    price = price_match.group(1) if price_match else "N/A"
    
    currency_match = re.search(r'(KSh)\s*[\d,]+', price_text)
    currency = currency_match.group(1) if currency_match else "KSh"  
    # print(currency)

    discount_elem = card.find('div', class_='bdg')
    discount = discount_elem.get_text(strip=True)
    # print(discount)
    
    datasets.append({
                        'name': name,
                        'brand': brand,                        
                        'currency':currency,
                        'price': price,
                        'discount': discount,
                    })

#store data as a dataframe/table using pandas
data_df = pd.DataFrame(datasets)

display(data_df)

,name,brand,currency,price,discount
0,Em ElectroMate Blow Dryer Hair Blower 2200W wi...,Em,KSh,999,41%
1,Em WD02 Water Dispenser Hot And Normal With St...,Em,KSh,"2,699",46%
2,"VILLAON V5606, 1.77"", 1000mAh, Feature Phone (...",VILLAON,KSh,860,14%
3,Kuhl KUHL K1 20000mAh Power Banks Portable Cha...,Kuhl,KSh,698,48%
4,"XIAOMI Redmi 15c, 6.9"", 4GB RAM + 128GB (Dual ...",XIAOMI,KSh,"12,000",20%
5,"Amtec 32R1S, 32"" Inch Smart Android Frameless ...",Amtec,KSh,"10,220",24%
6,Nunix 138L Double Door Fridge Energy Efficient...,Nunix,KSh,"24,035",6%
7,Rechargeab Touch Electric Water Dispenser Pump...,Generic,KSh,495,29%
8,Solarmax MX-316 Solar Max Portable Solar Kit L...,Solarmax,KSh,"1,990",34%
9,Solarmax 200Ah Solar Battery Deep Cycle Gel No...,Solarmax,KSh,"9,800",55%
